# 04 — F1 Prediction Engine

This notebook loads the existing trained models and processed historical dataset to build a reusable F1 race-winner prediction engine.

In [1]:

from pathlib import Path
import joblib
import pandas as pd

# Project paths
PROJECT_DIR = Path.cwd()

# If this notebook is running from the notebooks folder,
# move up one level to the project root.
if PROJECT_DIR.name.lower() == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

DATASET_PATH = PROJECT_DIR / "data" / "processed" / "expanded_model_dataset.csv"
MODELS_DIR = PROJECT_DIR / "models"

# Load the existing dataset
df = pd.read_csv(DATASET_PATH)

# Load the saved models
baseline_model = joblib.load(MODELS_DIR / "baseline_model.joblib")
logistic_model = joblib.load(MODELS_DIR / "logistic_regression_model.joblib")
random_forest_model = joblib.load(MODELS_DIR / "random_forest_model.joblib")

print("Dataset loaded:", df.shape)
print("Seasons available:", sorted(df["season"].unique()))

print("\nModels loaded successfully:")
print("1. Starting-grid baseline")
print("2. Logistic Regression")
print("3. Random Forest")

display(df.head())


Dataset loaded: (3458, 27)
Seasons available: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

Models loaded successfully:
1. Starting-grid baseline
2. Logistic Regression
3. Random Forest


,season,round,race_name,race_date,driver_id,driver_code,driver_number,driver_name,constructor_id,constructor_name,...,fastest_lap_rank,fastest_lap_time,position_num,won_race,driver_prior_mean_finish,driver_prior_win_rate,driver_recent5_mean_finish,constructor_prior_mean_finish,constructor_prior_win_rate,starting_position_feature
0,2018,1,Australian Grand Prix,2018-03-25,alonso,ALO,14,Fernando Alonso,mclaren,McLaren,...,7.0,1:26.978,5,0,NaN,NaN,NaN,NaN,NaN,10.0
1,2018,1,Australian Grand Prix,2018-03-25,bottas,BOT,77,Valtteri Bottas,mercedes,Mercedes,...,8.0,1:27.019,8,0,NaN,NaN,NaN,NaN,NaN,15.0
2,2018,1,Australian Grand Prix,2018-03-25,brendon_hartley,HAR,28,Brendon Hartley,toro_rosso,Toro Rosso,...,13.0,1:28.176,15,0,NaN,NaN,NaN,NaN,NaN,16.0
3,2018,1,Australian Grand Prix,2018-03-25,ericsson,ERI,9,Marcus Ericsson,sauber,Sauber,...,19.0,1:32.210,19,0,NaN,NaN,NaN,NaN,NaN,17.0
4,2018,1,Australian Grand Prix,2018-03-25,gasly,GAS,10,Pierre Gasly,toro_rosso,Toro Rosso,...,18.0,1:30.649,18,0,NaN,NaN,NaN,NaN,NaN,20.0


In [2]:

# Features used by each saved model

BASELINE_FEATURES = [
    "starting_position_feature"
]

HISTORY_FEATURES = [
    "starting_position_feature",
    "driver_prior_mean_finish",
    "driver_prior_win_rate",
    "driver_recent5_mean_finish",
    "constructor_prior_mean_finish",
    "constructor_prior_win_rate",
]

# Verify that the dataset contains all required columns
required_columns = set(BASELINE_FEATURES + HISTORY_FEATURES + ["won_race"])

missing_columns = required_columns - set(df.columns)

if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

print("Feature validation passed.")
print("\nStarting-grid baseline features:")
print(BASELINE_FEATURES)

print("\nHistorical model features:")
print(HISTORY_FEATURES)

print("\nTarget column: won_race")
print("Target is kept separate from model inputs.")



Feature validation passed.

Starting-grid baseline features:
['starting_position_feature']

Historical model features:
['starting_position_feature', 'driver_prior_mean_finish', 'driver_prior_win_rate', 'driver_recent5_mean_finish', 'constructor_prior_mean_finish', 'constructor_prior_win_rate']

Target column: won_race
Target is kept separate from model inputs.


In [3]:

import numpy as np

# Select one completed race for our first test
race = df[
    (df["season"] == 2025) &
    (df["race_name"] == "Australian Grand Prix")
].copy()

if race.empty:
    raise ValueError("The selected race was not found in the dataset.")

# Prepare the model inputs
X_baseline = race[BASELINE_FEATURES]
X_history = race[HISTORY_FEATURES]

# Generate each model's winner scores
race["baseline_score"] = baseline_model.predict_proba(
    X_baseline
)[:, 1]

race["logistic_score"] = logistic_model.predict_proba(
    X_history
)[:, 1]

race["random_forest_score"] = random_forest_model.predict_proba(
    X_history
)[:, 1]

# Normalize scores within this race
# These are relative probabilities, not yet calibrated probabilities.
score_columns = [
    "baseline_score",
    "logistic_score",
    "random_forest_score",
]

for column in score_columns:
    total = race[column].sum()

    if total <= 0:
        raise ValueError(f"Cannot normalize {column}: score total is zero.")

    race[column.replace("_score", "_probability")] = race[column] / total

# Display rankings from each model
def show_ranking(data, probability_column):
    result = data.sort_values(
        probability_column, ascending=False
    ).copy()

    result["predicted_probability"] = (
        result[probability_column] * 100
    ).round(2)

    return result[
        [
            "driver_name",
            "constructor_name",
            "predicted_probability",
            "won_race",
        ]
    ].rename(columns={
        "driver_name": "Driver",
        "constructor_name": "Constructor",
        "predicted_probability": "Win probability (%)",
        "won_race": "Actual winner (1 = yes)",
    }).reset_index(drop=True)

print("RACE:", race["race_name"].iloc[0], race["season"].iloc[0])

print("\nLOGISTIC REGRESSION RANKING")
display(show_ranking(race, "logistic_probability"))

print("\nRANDOM FOREST RANKING")
display(show_ranking(race, "random_forest_probability"))

print("\nSTARTING-GRID BASELINE RANKING")
display(show_ranking(race, "baseline_probability"))

# Verify each model's normalized probabilities
for column in [
    "baseline_probability",
    "logistic_probability",
    "random_forest_probability",
]:
    print(f"{column}: {race[column].sum():.6f}")


RACE: Australian Grand Prix 2025

LOGISTIC REGRESSION RANKING


,Driver,Constructor,Win probability (%),Actual winner (1 = yes)
0,Max Verstappen,Red Bull,34.26,0
1,Lando Norris,McLaren,29.88,1
2,Oscar Piastri,McLaren,14.82,0
3,George Russell,Mercedes,7.82,0
4,Charles Leclerc,Ferrari,5.28,0
5,Lewis Hamilton,Ferrari,4.39,0
6,Carlos Sainz,Williams,1.20,0
7,Yuki Tsunoda,RB F1 Team,1.10,0
8,Pierre Gasly,Alpine F1 Team,0.57,0
9,Alexander Albon,Williams,0.31,0



RANDOM FOREST RANKING


,Driver,Constructor,Win probability (%),Actual winner (1 = yes)
0,Lando Norris,McLaren,33.23,1
1,Max Verstappen,Red Bull,22.05,0
2,Lewis Hamilton,Ferrari,17.27,0
3,Oscar Piastri,McLaren,15.10,0
4,George Russell,Mercedes,4.20,0
5,Charles Leclerc,Ferrari,2.58,0
6,Alexander Albon,Williams,1.97,0
7,Carlos Sainz,Williams,0.94,0
8,Lance Stroll,Aston Martin,0.81,0
9,Yuki Tsunoda,RB F1 Team,0.62,0



STARTING-GRID BASELINE RANKING


,Driver,Constructor,Win probability (%),Actual winner (1 = yes)
0,Lando Norris,McLaren,36.96,1
1,Oscar Piastri,McLaren,24.83,0
2,Max Verstappen,Red Bull,15.69,0
3,George Russell,Mercedes,9.49,0
4,Yuki Tsunoda,RB F1 Team,5.58,0
5,Alexander Albon,Williams,3.22,0
6,Charles Leclerc,Ferrari,1.84,0
7,Lewis Hamilton,Ferrari,1.04,0
8,Pierre Gasly,Alpine F1 Team,0.59,0
9,Carlos Sainz,Williams,0.33,0


baseline_probability: 1.000000
logistic_probability: 1.000000
random_forest_probability: 1.000000


In [4]:

def predict_race(season, race_name):
    """
    Rank drivers for a historical race using the saved models.

    Returns a DataFrame with normalized model scores and
    the actual winner label for evaluation.
    """
    race_data = df[
        (df["season"] == season) &
        (df["race_name"] == race_name)
    ].copy()

    if race_data.empty:
        raise ValueError(
            f"No race found for {race_name}, season {season}."
        )

    # Generate model scores
    race_data["baseline_score"] = baseline_model.predict_proba(
        race_data[BASELINE_FEATURES]
    )[:, 1]

    race_data["logistic_score"] = logistic_model.predict_proba(
        race_data[HISTORY_FEATURES]
    )[:, 1]

    race_data["random_forest_score"] = random_forest_model.predict_proba(
        race_data[HISTORY_FEATURES]
    )[:, 1]

    # Normalize scores within the selected race
    for model_name in ["baseline", "logistic", "random_forest"]:
        score_col = f"{model_name}_score"
        probability_col = f"{model_name}_probability"
        total = race_data[score_col].sum()

        if total <= 0:
            raise ValueError(
                f"Cannot normalize scores for {model_name}."
            )

        race_data[probability_col] = race_data[score_col] / total

    return race_data.sort_values(
        "logistic_probability", ascending=False
    ).reset_index(drop=True)


# Test the reusable function
prediction = predict_race(2025, "Australian Grand Prix")

display(
    prediction[
        [
            "driver_name",
            "constructor_name",
            "logistic_probability",
            "random_forest_probability",
            "baseline_probability",
            "won_race",
        ]
    ].round(4)
)


,driver_name,constructor_name,logistic_probability,random_forest_probability,baseline_probability,won_race
0,Max Verstappen,Red Bull,0.3426,0.2205,0.1569,0
1,Lando Norris,McLaren,0.2988,0.3323,0.3696,1
2,Oscar Piastri,McLaren,0.1482,0.1510,0.2483,0
3,George Russell,Mercedes,0.0782,0.0420,0.0949,0
4,Charles Leclerc,Ferrari,0.0528,0.0258,0.0184,0
5,Lewis Hamilton,Ferrari,0.0439,0.1727,0.0104,0
6,Carlos Sainz,Williams,0.0120,0.0094,0.0033,0
7,Yuki Tsunoda,RB F1 Team,0.0110,0.0062,0.0558,0
8,Pierre Gasly,Alpine F1 Team,0.0057,0.0030,0.0059,0
9,Alexander Albon,Williams,0.0031,0.0197,0.0322,0


In [5]:

def predict_race(season, race_name, model_name="logistic"):
    """
    Predict and rank drivers for a race in the historical dataset.

    model_name: 'logistic', 'random_forest', or 'baseline'
    """
    models = {
        "logistic": (logistic_model, HISTORY_FEATURES),
        "random_forest": (random_forest_model, HISTORY_FEATURES),
        "baseline": (baseline_model, BASELINE_FEATURES),
    }

    if model_name not in models:
        raise ValueError(
            "model_name must be 'logistic', "
            "'random_forest', or 'baseline'."
        )

    race_data = df[
        (df["season"] == season) &
        (df["race_name"] == race_name)
    ].copy()

    if race_data.empty:
        raise ValueError(
            f"No race found for {race_name}, season {season}."
        )

    model, features = models[model_name]

    # The saved sklearn pipelines handle missing feature values
    scores = model.predict_proba(race_data[features])[:, 1]

    score_total = scores.sum()
    if score_total <= 0:
        raise ValueError("Model returned scores that cannot be normalized.")

    race_data["win_probability"] = scores / score_total
    race_data["win_probability_pct"] = (
        race_data["win_probability"] * 100
    ).round(2)

    race_data = race_data.sort_values(
        "win_probability", ascending=False
    ).reset_index(drop=True)

    race_data.insert(0, "predicted_rank", range(1, len(race_data) + 1))

    return race_data[
        [
            "predicted_rank",
            "driver_name",
            "constructor_name",
            "win_probability_pct",
            "won_race",
        ]
    ]


# Test model selection
display(predict_race(2025, "Australian Grand Prix", "logistic"))
display(predict_race(2025, "Australian Grand Prix", "random_forest"))
display(predict_race(2025, "Australian Grand Prix", "baseline"))


,predicted_rank,driver_name,constructor_name,win_probability_pct,won_race
0,1,Max Verstappen,Red Bull,34.26,0
1,2,Lando Norris,McLaren,29.88,1
2,3,Oscar Piastri,McLaren,14.82,0
3,4,George Russell,Mercedes,7.82,0
4,5,Charles Leclerc,Ferrari,5.28,0
5,6,Lewis Hamilton,Ferrari,4.39,0
6,7,Carlos Sainz,Williams,1.20,0
7,8,Yuki Tsunoda,RB F1 Team,1.10,0
8,9,Pierre Gasly,Alpine F1 Team,0.57,0
9,10,Alexander Albon,Williams,0.31,0


,predicted_rank,driver_name,constructor_name,win_probability_pct,won_race
0,1,Lando Norris,McLaren,33.23,1
1,2,Max Verstappen,Red Bull,22.05,0
2,3,Lewis Hamilton,Ferrari,17.27,0
3,4,Oscar Piastri,McLaren,15.10,0
4,5,George Russell,Mercedes,4.20,0
5,6,Charles Leclerc,Ferrari,2.58,0
6,7,Alexander Albon,Williams,1.97,0
7,8,Carlos Sainz,Williams,0.94,0
8,9,Lance Stroll,Aston Martin,0.81,0
9,10,Yuki Tsunoda,RB F1 Team,0.62,0


,predicted_rank,driver_name,constructor_name,win_probability_pct,won_race
0,1,Lando Norris,McLaren,36.96,1
1,2,Oscar Piastri,McLaren,24.83,0
2,3,Max Verstappen,Red Bull,15.69,0
3,4,George Russell,Mercedes,9.49,0
4,5,Yuki Tsunoda,RB F1 Team,5.58,0
5,6,Alexander Albon,Williams,3.22,0
6,7,Charles Leclerc,Ferrari,1.84,0
7,8,Lewis Hamilton,Ferrari,1.04,0
8,9,Pierre Gasly,Alpine F1 Team,0.59,0
9,10,Carlos Sainz,Williams,0.33,0


In [6]:

# Evaluate the reusable prediction function across 2025 races

test_season = 2025
season_data = df[df["season"] == test_season]

race_list = (
    season_data[["round", "race_name"]]
    .drop_duplicates()
    .sort_values("round")
)

evaluation_rows = []

for _, race_info in race_list.iterrows():
    race_round = race_info["round"]
    race_name = race_info["race_name"]

    for model_name in ["logistic", "random_forest", "baseline"]:
        ranking = predict_race(test_season, race_name, model_name)

        predicted_winner = ranking.iloc[0]
        actual_winner = ranking[ranking["won_race"] == 1]

        evaluation_rows.append({
            "season": test_season,
            "round": race_round,
            "race_name": race_name,
            "model": model_name,
            "predicted_winner": predicted_winner["driver_name"],
            "actual_winner": (
                actual_winner.iloc[0]["driver_name"]
                if not actual_winner.empty else None
            ),
            "correct": int(predicted_winner["won_race"] == 1),
        })

evaluation = pd.DataFrame(evaluation_rows)

summary = (
    evaluation.groupby("model")
    .agg(
        races_evaluated=("race_name", "nunique"),
        correct_winners=("correct", "sum"),
        top1_accuracy=("correct", "mean"),
    )
    .reset_index()
)

summary["top1_accuracy_pct"] = (
    summary["top1_accuracy"] * 100
).round(2)

display(summary)


,model,races_evaluated,correct_winners,top1_accuracy,top1_accuracy_pct
0,baseline,24,16,0.666667,66.67
1,logistic,24,12,0.500000,50.00
2,random_forest,24,14,0.583333,58.33
